<a href="https://colab.research.google.com/github/MightyCrimsonX/Kohya-Anima-Trainer-Mighty/blob/main/MightyCrimson_LoraAnimaTrainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌟 Anima Preview Lora Trainer (WIP) por *Mighty Crimson*

Para Taggear el dataset, recomiendo usar el Dataset maker por [WhiteZ](https://colab.research.google.com/github/gwhitez/Lora-Trainer-XL/blob/main/Dataset_Maker_By_WhiteZ.ipynb?authuser=1#scrollTo=xEsqOglcc6hA)

This colab is based on the work of [Kohya-ss](https://github.com/kohya-ss/sd-scripts) and [Linaqruf](https://github.com/Linaqruf/kohya-trainer). Thank you!


### ⭕ Disclaimer
The purpose of this document is to research bleeding-edge technologies in the field of machine learning inference.  
Please read and follow the [Google Colab guidelines](https://research.google.com/colaboratory/faq.html) and its [Terms of Service](https://research.google.com/colaboratory/tos_v3.html).

In [ ]:
import os, re, sys, toml
from pathlib import Path
from time import time
import time
from IPython.display import Markdown, display, HTML, clear_output
from huggingface_hub.utils import disable_progress_bars
import logging
from IPython import get_ipython
import subprocess

disable_progress_bars()
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("ACCELERATE_DISABLE_RICH_PROGRESS", "1")
os.environ.setdefault("ACCELERATE_DISABLE_PROGRESS_BAR", "1")
os.environ.setdefault("TQDM_MININTERVAL", "2")
os.environ.setdefault("TQDM_NOPOS", "1")
logging.getLogger("accelerate").setLevel(logging.WARNING)
logging.getLogger("accelerate.tracking").setLevel(logging.ERROR)
logging.getLogger("lightning").setLevel(logging.WARNING)
logging.getLogger("tqdm").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.WARNING)

def _run_cmd(command: str) -> None:
    ip = get_ipython()
    if ip is not None:
        ip.system(command)
    else:
        subprocess.run(command, shell=True, check=True)

#auto off
#@markdown ### 💤 Auto Off
#@markdown *Si esta opción está marcada, el entorno se desconectará automáticamente una vez finalice el entrenamiento.*
auto_off = True  # @param {type: "boolean"}
if auto_off:
    print("\033[96mAuto Off encendido\033[0m")
else:
    print("\033[96mAuto Off apagado\033[0m")

root_dir = "/content"
kohya_dir = os.path.join(root_dir, "sd-scripts")
models_dir = os.path.join(root_dir, "models")
downloads_dir = os.path.join(root_dir, "downloads")

# These carry information from past executions
if "model_url" in globals():
  old_model_url = model_url
else:
  old_model_url = None
if "dependencies_installed" not in globals():
  dependencies_installed = False
if "model_file" not in globals():
  model_file = None

if "custom_dataset" not in globals():
  custom_dataset = None
if "override_dataset_config_file" not in globals():
  override_dataset_config_file = None
if "continue_from_lora" not in globals():
  continue_from_lora = ""
if "override_config_file" not in globals():
  override_config_file = None

COMMIT = "main"
LOWRAM = False
LOAD_TRUNCATED_IMAGES = True
BETTER_EPOCH_NAMES = True
FIX_DIFFUSERS = True
FIX_WANDB_WARNING = True

#@title ## 🚩 Start Here (Anima LoRA Training)

#@markdown ### ▶️ Setup
#@markdown El nombre de tu proyecto será el mismo que el de la carpeta que contiene tus imágenes. No se permiten espacios, puedes usar `guión bajo` si el nombre es muy largo.
project_name = "" #@param {type:"string"}
project_name = project_name.strip()
#@markdown La estructura de carpetas no importa y es puramente por comodidad. Asegúrate de elegir siempre el mismo.
folder_structure = "Organize by project (MyDrive/Loras/project_name/dataset)" #@param ["Organize by category (MyDrive/lora_training/datasets/project_name)", "Organize by project (MyDrive/Loras/project_name/dataset)"]

#@markdown #### Modelo Anima
#@markdown Selecciona el modelo Anima DiT base. También puedes pegar un enlace de descarga o proporcionar una ruta local.
training_model = "Anima-1.0-Base" # @param ["Anima-Preview-2","Anima-Preview-3","Anima-1.0-Base"]
optional_custom_training_model = "" #@param {type:"string"}
#@markdown Activa esta opción para utilizar un modelo personalizado desde la URL anterior en lugar del modelo base seleccionado.
use_optional_custom_training_model = False #@param {type:"boolean"}
#@markdown Ruta al LLM Adapter (si no está incluido en el modelo DiT).
llm_adapter_path = "" #@param {type:"string"}
#@markdown Ruta al T5 Tokenizer (si se desea usar uno distinto al incluido en configs/t5_old/).
t5_tokenizer_path = "" #@param {type:"string"}

wandb_key = "" #@param {type:"string"}

custom_model_selected = use_optional_custom_training_model and len(optional_custom_training_model) > 0

# --- Anima Model URLs ---
if custom_model_selected:
  model_url = optional_custom_training_model
elif "Anima-Preview-2" in training_model:
  model_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-preview2.safetensors"
  model_file = os.path.join(models_dir, "anima-preview2.safetensors")
elif "Anima-Preview-3" in training_model:
  model_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-preview3-base.safetensors"
  model_file = os.path.join(models_dir, "anima-preview3-base.safetensors")
elif "Anima-1.0-Base" in training_model:
  model_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-base-v1.0.safetensors"
  model_file = os.path.join(models_dir, "anima-base-v1.0.safetensors")
else:
  model_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-preview2.safetensors"
  model_file = os.path.join(models_dir, "anima_preview_dit.safetensors")

model_url = model_url.strip()

#@markdown #### Rutas de componentes Anima
#@markdown Qwen3-0.6B y Qwen-Image VAE se descargan automáticamente en la carpeta de modelos.
qwen3_path = os.path.join(models_dir, "qwen_3_06b_base.safetensors")
anima_vae_path = os.path.join(models_dir, "qwen_image_vae.safetensors")
vae_file = anima_vae_path

qwen3_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors"
anima_vae_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors"

#@markdown ### ▶️ Processing
#@markdown Por defecto la resolución para Anima es 1024. Otras resoluciones posibles son 896 o 768.
resolution = 768 #@param {type:"dropdown", min:512, max:1536, step:128}
#@markdown Activa `Flip Aug` si tu dataset es pequeño, volteará tus imágenes (modo espejo).
flip_aug = False #@param {type:"boolean"}
caption_extension = ".txt" # @param [".txt",".caption"]
#@markdown Mezcla etiquetas, mejora el aprendizaje. Una etiqueta de activación va al comienzo de cada archivo de texto y no se mezclará.
shuffle_tags = True #@param {type:"boolean"}
shuffle_caption = shuffle_tags
caption_tag_dropout_rate = 0.05 #@param {type:"dropdown", min:0, max:0.1, step:0.01}
activation_tags = "1" #@param ["0","1","2","3"]
keep_tokens = int(activation_tags)

#@markdown ### ▶️ Steps
#@markdown Tus imágenes se repetirán esta cantidad de veces durante el entrenamiento.
num_repeats = 10 #@param {type:"number"}
#@markdown Elige cuánto tiempo quieres entrenar.
preferred_unit = "Epochs" #@param ["Epochs", "Steps"]
how_many = 15 #@param {type:"number"}
max_train_epochs = how_many if preferred_unit == "Epochs" else None
max_train_steps = how_many if preferred_unit == "Steps" else None
#@markdown Guardar más épocas te permitirá comparar el progreso de tu LoRA.
save_every_n_epochs = 1 #@param {type:"number"}
keep_only_last_n_epochs = 7 #@param {type:"number"}
if not save_every_n_epochs:
  save_every_n_epochs = max_train_epochs
if not keep_only_last_n_epochs:
  keep_only_last_n_epochs = max_train_epochs

#@markdown ### ▶️ Learning
#@markdown La tasa de aprendizaje para Anima. Valor recomendado: 1e-4 (para alpha=1.0 por defecto).
unet_lr = 5e-05 #@param {type:"number"}
#@markdown Learning rate del text encoder Qwen3. Se recomienda la mitad del unet_lr o menos. Ponlo en 0 para no entrenar el text encoder.
text_encoder_lr = 0 #@param {type:"number"}
#@markdown El scheduler es el algoritmo que guía la tasa de aprendizaje.
lr_scheduler = "rex" # @param ["constant","cosine","cosine_with_restarts","constant_with_warmup","linear","polynomial","rex"]
lr_scheduler_number = 1 #@param {type:"number"}
#@markdown Pasos de warmup como proporción del total.
lr_warmup_ratio = 0.05 #@param {type:"slider", min:0.0, max:0.2, step:0.01}
lr_warmup_steps = 100 #@param {type:"number"}
#@markdown `ip_noise_gamma` ajusta el ruido aleatorio. Nota: min_snr_gamma NO es compatible con Anima (usa Rectified Flow).
ip_noise_gamma_enabled = True #@param {type:"boolean"}
ip_noise_gamma = 0.05 #@param {type:"slider", min:0.05, max:0.1, step:0.01}

# ### ▶️ Text Encoder LoRA
# Activa `network_train_unet_only` para entrenar SOLO el DiT (sin LoRA del text encoder Qwen3). Recomendado activar si usas `cache_text_encoder_outputs`.
network_train_unet_only = True

#@markdown ### ▶️ Structure (Anima LoRA)
#@markdown Valores recomendados: network_dim=8, network_alpha=4.
lora_type = "LoRA" # @param ["LoRA","LoCon"]

network_dim = 8 #@param {type:"number", min:1, max:128, step:1}
network_alpha = 4 #@param {type:"number", min:1, max:128, step:1}
#@markdown Los siguientes dos valores solo se aplican a las capas adicionales de LoCon.
conv_dim = 16 #@param {type:"number", min:1, max:32, step:1}
conv_alpha = 16 #@param {type:"number", min:1, max:32, step:1}
network_args = None
network_module = "networks.lora_anima"
if lora_type.lower() == "locon":
    network_module = "lycoris.kohya"
    network_args = [
      "algo=locon",
      f"conv_dim={int(conv_dim)}",
      f"conv_alpha={int(conv_alpha)}"
    ]

#@markdown ### ▶️ Anima-Specific Parameters
#@markdown Método de muestreo de timesteps. `sigmoid` es el valor por defecto y funciona bien en general.
timestep_sampling = "sigmoid" #@param ["sigma", "uniform", "sigmoid", "shift", "flux_shift"]
#@markdown Shift para la distribución de timesteps en Rectified Flow. Solo aplica cuando timestep_sampling='shift'.
discrete_flow_shift = 3.0 #@param {type:"number"}
#@markdown Factor de escala para sigmoid/shift/flux_shift timestep sampling.
sigmoid_scale = 1.0 #@param {type:"number"}
#Longitud máxima de tokens para Qwen3.
qwen3_max_token_length = 512
#Longitud máxima de tokens para T5.
t5_max_token_length = 512
#@markdown Esquema de ponderación de pérdida por timestep.
weighting_scheme = "uniform" #@param ["uniform", "sigma_sqrt", "cosmap", "none"]

#@markdown ### ▶️ Anima Memory Optimization
#Número de bloques Transformer para intercambiar entre CPU y GPU (0 para desactivar). Máx 26 para Anima-Preview.
blocks_to_swap = 0
#Chunk size para Qwen-Image VAE. Reduce VRAM a costa de velocidad.
vae_chunk_size = 0
#@markdown Desactivar caché interno del VAE para reducir VRAM.
vae_disable_cache = True #@param {type:"boolean"}
#Offload de checkpoints de activación a CPU. No se puede usar con blocks_to_swap.
unsloth_offload_checkpointing = False

#@markdown ### ▶️ LLM Adapter & Regex Module Control
#Activa para aplicar LoRA también al LLM Adapter.
train_llm_adapter = False
#@markdown Patrones de exclusión / inclusión de módulos (regex, separados por comas).
exclude_patterns = "" #@param {type:"string"}
exclude_patterns = exclude_patterns.strip()
include_patterns = "" #@param {type:"string"}
include_patterns = include_patterns.strip()
#@markdown Dims y lrs por módulo (regex). Formato: `.*self_attn.*=8,.*cross_attn.*=4`
network_reg_dims = "" #@param {type:"string"}
network_reg_dims = network_reg_dims.strip()
network_reg_lrs = "" #@param {type:"string"}
network_reg_lrs = network_reg_lrs.strip()

#@markdown ### ▶️ Training
#@markdown Ajuste estos parámetros según la configuración de su entorno.
train_batch_size = 2 #@param {type:"slider", min:1, max:16, step:1}
#Implementación de atención a usar. `torch` es el valor por defecto.
attn_mode = "torch"
#@markdown Precisión mixta para el entrenamiento (La T4 (Colab free) No soporta bf16!).
precision = "mixed fp16" #@param ["full fp16", "full bf16", "mixed fp16", "mixed bf16"]
#@markdown Cachear latentes del VAE para liberar VRAM.
cache_latents = True #@param {type:"boolean"}
cache_latents_to_disk = True #@param {type:"boolean"}
#@markdown Cachear salidas del text encoder Qwen3.
cache_text_encoder_outputs  = False  # @param {type:"boolean"}

mixed_precision = "no"
if "fp16" in precision:
  mixed_precision = "fp16"
elif "bf16" in precision:
  mixed_precision = "bf16"
full_precision = "full" in precision

#@markdown ### ▶️ Advanced
#@markdown El optimizador utilizado para el entrenamiento.
optimizer = "CAME" #@param ["AdamW8bit", "AdamW8bitKahan", "Prodigy", "DAdaptation", "Emosens", "DadaptAdam", "DadaptLion", "AdamW", "Lion", "SGDNesterov", "SGDNesterov8bit", "AdaFactor", "CAME"]
#@markdown Si se selecciona Dadapt o Prodigy y se marca la casilla recomendada, se aplicarán valores optimizados.
recommended_values = True #@param {type:"boolean"}
#@markdown Alternativamente, establezca sus propios argumentos de optimizador separados por espacios.
optimizer_args = "" #@param {type:"string"}
optimizer_args = [a.strip() for a in optimizer_args.split(' ') if a]
#@markdown Tipo de pérdida (loss function). `l2` es el valor por defecto para Anima.
loss_type = "l2" #@param ["l1", "l2", "huber", "smooth_l1"]
#@markdown ### ▶️ Sample Image Generator 🖼️
#@markdown Genera imágenes de muestra durante el entrenamiento para monitorear el progreso visual del LoRA.
enable_sample_generation = False #@param {type:"boolean"}
#@markdown Genera muestras cada N épocas.
sample_every_n_epochs = 1 #@param {type:"number"}
#@markdown Genera una muestra antes de iniciar el entrenamiento.
sample_at_first = False #@param {type:"boolean"}
#@markdown **Prompt positivo** para las imágenes de muestra.
sample_positive_prompt = "masterpiece, best quality," #@param {type:"string"}
#@markdown **Prompt negativo** para las imágenes de muestra.
sample_negative_prompt = "low quality, worst quality" #@param {type:"string"}
#@markdown Resolución de las imágenes de muestra.
sample_resolution = "768x1344" #@param ["832x1216", "1216x832", "768x1344", "1344x768", "768x1024", "1024x768", "1024x1024", "896x1152", "1152x896"]
#@markdown Semilla para la generación (se usará la misma en todas las muestras).
sample_seed = 42 #@param {type:"number"}
#@markdown CFG Scale para la generación de muestras.
sample_cfg_scale = 4.5 #@param {type:"number"}
#@markdown Número de pasos de generación para las muestras.
sample_steps = 23 #@param {type:"number"}
#@markdown Sampler/Scheduler para la generación de muestras.
sample_sampler = "euler_a" #@param ["euler_a", "euler", "dpm++_2m_karras", "dpm++_2m", "dpm++_sde_karras", "ddim"]



if recommended_values:
  if any(opt in optimizer.lower() for opt in ["dadapt", "prodigy"]):
    unet_lr = 1.0
    text_encoder_lr = 1.0
    full_precision = False
    network_alpha = network_dim
  if optimizer == "Prodigy":
    optimizer_args = ["d_coef=1", "use_bias_correction=True", "safeguard_warmup=True", "weight_decay=0.01", "decouple=True"]
  elif optimizer == "AdamW8bit":
    optimizer_args = ["weight_decay=0.1", "betas=[0.9,0.99]"]
  elif optimizer == "AdamW8bitKahan":
    optimizer_args = ["weight_decay=0.04"]
  elif optimizer == "AdaFactor":
    optimizer_args = ["scale_parameter=False", "relative_step=False", "warmup_init=False"]
  elif optimizer == "CAME":
    optimizer_args = ["weight_decay=0.04"]

if optimizer == "CAME":
  optimizer = "LoraEasyCustomOptimizer.came.CAME"
  optimizer_args = ["weight_decay=0.04"]


if optimizer == "AdamW8bitKahan":
  optimizer = "LoraEasyCustomOptimizer.adam.AdamW8bitKahan"
  optimizer_args = ["weight_decay=0.04"]

if optimizer == "Emosens":
  optimizer = "optimizer.emosens.EmoSens"
  ip_noise_gamma_enabled = False

lr_scheduler_type = None
lr_scheduler_args_list = []
lr_scheduler_num_cycles = lr_scheduler_number
lr_scheduler_power = lr_scheduler_number

if "rex" in lr_scheduler:
  lr_scheduler = "cosine"
  lr_scheduler_type = "LoraEasyCustomOptimizer.RexAnnealingWarmRestarts.RexAnnealingWarmRestarts"
  lr_scheduler_args_list = ["min_lr=1e-9", "gamma=0.9", "d=0.9"]

seed = 42
gradient_accumulation_steps = 1
bucket_reso_steps = 64
min_bucket_reso = 256
max_bucket_reso = 1536

#@markdown ### ▶️ Ready
#@markdown Ahora puedes ejecutar esta celda para entrenar tu LoRA de Anima. ¡Buena suerte!

for required_dir in (models_dir, downloads_dir):
  os.makedirs(required_dir, exist_ok=True)

venv_python = "python"
train_network = os.path.join(kohya_dir, "anima_train_network.py")

if "/Loras" in folder_structure:
  main_dir      = os.path.join(root_dir, "drive/MyDrive/Loras")
  log_folder    = os.path.join(main_dir, "_logs")
  config_folder = os.path.join(main_dir, project_name)
  images_folder = os.path.join(main_dir, project_name, "dataset")
  output_folder = os.path.join(main_dir, project_name, "output")
else:
  main_dir      = os.path.join(root_dir, "drive/MyDrive/lora_training")
  images_folder = os.path.join(main_dir, "datasets", project_name)
  output_folder = os.path.join(main_dir, "output", project_name)
  config_folder = os.path.join(main_dir, "config", project_name)
  log_folder    = os.path.join(main_dir, "log")

config_file = os.path.join(config_folder, "training_config.toml")
dataset_config_file = os.path.join(config_folder, "dataset_config.toml")

# --- Sample Image Generation Paths ---
samples_folder = os.path.join(main_dir, project_name, "samples_img")
sample_prompt_file = os.path.join(config_folder, "sample_prompts.txt")

def install_trainer():
  global installed
  if getattr(install_trainer, "_run", False):
    return

  display(HTML("<h2 style='color: yellow;'>Clonando sd-scripts y descargando dependencias</h2>"))

  # Ensure uv is installed
  _run_cmd("pip install uv")

  if not os.path.exists(kohya_dir):
    _run_cmd(f"git clone https://github.com/kohya-ss/sd-scripts.git {kohya_dir}")


  os.chdir(root_dir)
  !git clone https://github.com/MightyCrimsonX/LoRA_Easy_Training_scripts_Backend.git
  !git clone https://github.com/muooon/EmoSens.git

  os.chdir(kohya_dir)
  _run_cmd("uv pip install --system -r requirements.txt")
  _run_cmd("sudo apt-get install aria2 -q")
  _run_cmd("uv pip install -U -e /content/LoRA_Easy_Training_scripts_Backend/custom_scheduler/.")
  _run_cmd("uv pip install -U --no-deps torchao~=0.13.0 --extra-index-url https://download.pytorch.org/whl/cu128 --no-progress")
  _run_cmd("uv pip install -U --force-reinstall --no-deps git+https://github.com/67372a/RamTorch")
  _run_cmd("uv pip install lycoris-lora")
  _run_cmd("uv pip install torch~=2.7.1 torchvision~=0.22.1 --index-url https://download.pytorch.org/whl/cu128 --no-progress")
  _run_cmd("uv pip install -U --no-deps xformers==0.0.31.post1 --index-url https://download.pytorch.org/whl/cu128")
  _run_cmd("uv pip install -U --force-reinstall --no-deps git+https://github.com/67372a/customized-optimizers")
  _run_cmd("uv pip install adv_optm~=2.2.3")
  !mv /content/EmoSens/optimizer /content/sd-scripts


  if LOAD_TRUNCATED_IMAGES:
    _run_cmd("sed -i 's/from PIL import Image/from PIL import Image, ImageFile\n ImageFile.LOAD_TRUNCATED_IMAGES=True/g' library/train_util.py")
  if BETTER_EPOCH_NAMES:
    _run_cmd("sed -i 's/{:06d}/{:02d}/g' library/train_util.py")
    train_network_path = Path(kohya_dir) / "train_network.py"
    if train_network_path.exists():
      text = train_network_path.read_text()
      pattern = re.compile(r'"-\{:[0-9]+d\}\.".format\(([^)]+)\)\s*\+\s*args\.save_model_as')
      def _repl(m): return 'f"-{' + m.group(1).strip() + ':02d}." + args.save_model_as'
      new_text, count = pattern.subn(_repl, text)
      if count: train_network_path.write_text(new_text)

  if FIX_WANDB_WARNING:
    _run_cmd("sed -i 's/accelerator.log(logs, step=epoch + 1)//g' train_network.py")

  os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
  os.environ["BITSANDBYTES_NOWELCOME"] = "1"
  os.environ["SAFETENSORS_FAST_GPU"] = "1"
  os.environ["PYTHONWARNINGS"] = "ignore"
  os.chdir(root_dir)
  install_trainer._run = True

def validate_dataset():
  global lr_warmup_steps, lr_warmup_ratio, caption_extension, keep_tokens, model_url
  supported_types = (".png", ".jpg", ".jpeg", ".webp", ".bmp")

  print("\n 💿 Checking dataset...")
  if not project_name.strip() or any(c in project_name for c in " .()\"'\\/"):
    print("💥 Error: Elija un nombre de proyecto válido.")
    return False

  if custom_dataset:
    try:
      datconf = toml.loads(custom_dataset)
      datasets = [d for d in datconf["datasets"][0]["subsets"]]
    except:
      print("💥 Error: El conjunto de datos personalizado no es válido.")
      return False
    reg = [d.get("image_dir") for d in datasets if d.get("is_reg", False)]
    datasets_dict = {d["image_dir"]: d["num_repeats"] for d in datasets}
    folders = datasets_dict.keys()
    files = [f for folder in folders for f in os.listdir(folder)]
    images_repeats = {folder: (len([f for f in os.listdir(folder) if f.lower().endswith(supported_types)]), datasets_dict[folder]) for folder in folders}
  else:
    reg = []
    folders = [images_folder]
    if os.path.exists(images_folder):
      files = os.listdir(images_folder)
    else:
      print(f"💥 Error: La carpeta {images_folder.replace('/content/drive/', '')} no existe.")
      return False
    images_repeats = {images_folder: (len([f for f in files if f.lower().endswith(supported_types)]), num_repeats)}

  for folder in folders:
    if not os.path.exists(folder):
      print(f"💥 Error: La carpeta {folder.replace('/content/drive/', '')} no existe.")
      return False
  for folder, (img, rep) in images_repeats.items():
    if not img:
      print(f"💥 Error: La carpeta {folder.replace('/content/drive/', '')} está vacía.")
      return False

  if continue_from_lora and not (continue_from_lora.endswith(".safetensors") and os.path.exists(continue_from_lora)):
    print(f"💥 Error: ruta no válida de LoRA existente.")
    return False

  pre_steps_per_epoch = sum(img*rep for (img, rep) in images_repeats.values())
  steps_per_epoch = pre_steps_per_epoch/train_batch_size
  total_steps = max_train_steps or int(max_train_epochs*steps_per_epoch)
  estimated_epochs = int(total_steps/steps_per_epoch)
  lr_warmup_steps = int(total_steps*lr_warmup_ratio)

  for folder, (img, rep) in images_repeats.items():
    print("📁"+folder.replace("/content/drive/", "") + (" (Regularization)" if folder in reg else ""))
    print(f"📈 Se encontró {img} imágenes con {rep} repeticiones, total {img*rep} pasos.")
  print(f"📉 Divide {pre_steps_per_epoch} pasos por {train_batch_size} batch size para obtener {steps_per_epoch} pasos/epoch.")
  if max_train_epochs:
    print(f"🔮 Habrá {max_train_epochs} epochs para aprox {total_steps} pasos.")
  else:
    print(f"🔮 Habrá {total_steps} pasos.")

  if total_steps > 20000:
    print("⚠️ Total de pasos alto. Asegúrese de que es lo que desea.")

  return True

def create_config():
  global dataset_config_file, config_file, model_file

  _network_args = []
  if train_llm_adapter:
    _network_args.append("train_llm_adapter=True")
  if exclude_patterns:
    _network_args.append(f"exclude_patterns=['{exclude_patterns}']")
  if include_patterns:
    _network_args.append(f"include_patterns=['{include_patterns}']")
  if network_reg_dims:
    _network_args.append(f"network_reg_dims={network_reg_dims}")
  if network_reg_lrs:
    _network_args.append(f"network_reg_lrs={network_reg_lrs}")

   # --- Create sample prompt file if enabled ---
  if enable_sample_generation:
    os.makedirs(samples_folder, exist_ok=True)
    sample_w, sample_h = sample_resolution.split("x")
    with open(sample_prompt_file, "w") as spf:
      prompt_line = f"{sample_positive_prompt} --n {sample_negative_prompt} --w {sample_w} --h {sample_h} --d {int(sample_seed)} --l {sample_cfg_scale} --s {int(sample_steps)}"
      spf.write(prompt_line + "\n")

  if override_config_file:
    config_file = override_config_file
  else:
    config_dict = {
      "network_arguments": {
        "unet_lr": unet_lr,
        "text_encoder_lr": text_encoder_lr if not cache_text_encoder_outputs else 0,
        "network_dim": network_dim,
        "network_alpha": network_alpha,
        "network_module": network_module,
        "network_args": _network_args if _network_args else None,
        "network_train_unet_only": network_train_unet_only or cache_text_encoder_outputs,
        "network_weights": continue_from_lora or None
      },
      "optimizer_arguments": {
        "learning_rate": unet_lr,
        "lr_scheduler": lr_scheduler,
        "lr_scheduler_type": lr_scheduler_type,
        "lr_scheduler_args": lr_scheduler_args_list,
        "lr_scheduler_num_cycles": lr_scheduler_num_cycles if lr_scheduler == "cosine_with_restarts" else None,
        "lr_scheduler_power": lr_scheduler_power if lr_scheduler == "polynomial" else None,
        "lr_warmup_steps": lr_warmup_steps if lr_scheduler not in ("cosine", "constant") else None,
        "optimizer_type": optimizer,
        "optimizer_args": optimizer_args or None,
        "loss_type": loss_type,
        "max_grad_norm": 1.0,
      },
      "training_arguments": {
        "lowram": LOWRAM,
        "pretrained_model_name_or_path": model_file,
        "qwen3": qwen3_path,
        "vae": vae_file,
        "llm_adapter_path": llm_adapter_path if llm_adapter_path else None,
        "t5_tokenizer_path": t5_tokenizer_path if t5_tokenizer_path else None,
        "max_train_steps": max_train_steps,
        "max_train_epochs": max_train_epochs,
        "train_batch_size": train_batch_size,
        "seed": seed,
        "timestep_sampling": timestep_sampling,
        "discrete_flow_shift": discrete_flow_shift,
        "sigmoid_scale": sigmoid_scale,
        "weighting_scheme": weighting_scheme,
        "qwen3_max_token_length": qwen3_max_token_length,
        "t5_max_token_length": t5_max_token_length,
        "attn_mode": attn_mode if attn_mode != "torch" else None,
        "split_attn": True if attn_mode == "xformers" else None,
        "ip_noise_gamma": ip_noise_gamma if ip_noise_gamma_enabled else None,
        "gradient_checkpointing": True,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "persistent_data_loader_workers": True,
        "mixed_precision": mixed_precision,
        "full_fp16": mixed_precision == "fp16" and full_precision,
        "full_bf16": mixed_precision == "bf16" and full_precision,
        "cache_latents": cache_latents,
        "cache_latents_to_disk": cache_latents_to_disk,
        "cache_text_encoder_outputs": cache_text_encoder_outputs,
        "blocks_to_swap": blocks_to_swap if blocks_to_swap > 0 else None,
        "vae_chunk_size": vae_chunk_size if vae_chunk_size > 0 else None,
        "vae_disable_cache": vae_disable_cache or None,
        "unsloth_offload_checkpointing": unsloth_offload_checkpointing or None,
      },
      "saving_arguments": {
        "save_precision": "fp16",
        "save_model_as": "safetensors",
        "save_every_n_epochs": save_every_n_epochs,
        "save_last_n_epochs": keep_only_last_n_epochs,
        "output_name": project_name,
        "output_dir": output_folder,
        "log_prefix": project_name,
        "logging_dir": log_folder,
        "wandb_api_key": wandb_key or None,
        "log_with": "wandb" if wandb_key else None,
      },
      "sample_arguments": {
        "sample_every_n_epochs": sample_every_n_epochs if enable_sample_generation else None,
        "sample_at_first": sample_at_first if enable_sample_generation else None,
        "sample_prompts": sample_prompt_file if enable_sample_generation else None,
        "sample_sampler": sample_sampler if enable_sample_generation else None,
      }
    }

    for key in config_dict:
      if isinstance(config_dict[key], dict):
        config_dict[key] = {k: v for k, v in config_dict[key].items() if v is not None}

    with open(config_file, "w") as f:
      f.write(toml.dumps(config_dict))

  if override_dataset_config_file:
    dataset_config_file = override_dataset_config_file
  else:
    dataset_config_dict = {
      "general": {
        "resolution": resolution,
        "shuffle_caption": shuffle_caption and not cache_text_encoder_outputs,
        "keep_tokens": keep_tokens,
        "flip_aug": flip_aug,
        "caption_extension": caption_extension,
        "enable_bucket": True,
        "bucket_reso_steps": bucket_reso_steps,
        "min_bucket_reso": min_bucket_reso,
        "max_bucket_reso": max_bucket_reso,
      },
      "datasets": toml.loads(custom_dataset)["datasets"] if custom_dataset else [
        {
          "subsets": [
            {
              "num_repeats": num_repeats,
              "image_dir": images_folder,
              "class_tokens": None if caption_extension else project_name,
              "caption_tag_dropout_rate": caption_tag_dropout_rate
            }
          ]
        }
      ]
    }

    for key in dataset_config_dict:
      if isinstance(dataset_config_dict[key], dict):
        dataset_config_dict[key] = {k: v for k, v in dataset_config_dict[key].items() if v is not None}

    with open(dataset_config_file, "w") as f:
      f.write(toml.dumps(dataset_config_dict))

def download_anima_components() -> bool:
  os.makedirs(models_dir, exist_ok=True)

  if custom_model_selected:
    print(f"📁 Usando modelo de URL: {model_url}")
    if not os.path.exists(model_file):
      _run_cmd(f"aria2c '{model_url}' --console-log-level=warn -c -s 16 -x 16 -k 10M -d {models_dir} -o '{os.path.basename(model_file)}'")
  else:
    if not os.path.exists(model_file):
      print(f"🌐 Descargando Anima DiT...")
      _run_cmd(f"aria2c '{model_url}' --console-log-level=warn -c -s 16 -x 16 -k 10M -d {models_dir} -o '{os.path.basename(model_file)}'")

  if not os.path.exists(qwen3_path):
    print(f"🌐 Descargando Qwen3-0.6B text encoder...")
    _run_cmd(f"aria2c '{qwen3_url}' --console-log-level=warn -c -s 16 -x 16 -k 10M -d {models_dir} -o '{os.path.basename(qwen3_path)}'")

  if not os.path.exists(anima_vae_path):
    print(f"🌐 Descargando Qwen-Image VAE...")
    _run_cmd(f"aria2c '{anima_vae_url}' --console-log-level=warn -c -s 16 -x 16 -k 10M -d {models_dir} -o '{os.path.basename(anima_vae_path)}'")

  return True

def calculate_rex_steps():
  global max_train_steps
  if 'max_train_steps' in globals() and max_train_steps:
    calculated_max_steps = max_train_steps
  else:
    with open(dataset_config_file, "r") as f:
      subsets = toml.load(f)["datasets"][0]["subsets"]
    supported_types = [".png", ".jpg", ".jpeg", ".webp", ".bmp"]
    total_images_with_repeats = 0
    for subset in subsets:
      image_dir = Path(subset["image_dir"])
      if not image_dir.exists(): continue
      subset_image_count = len([f for f in image_dir.iterdir() if f.suffix.lower() in supported_types])
      total_images_with_repeats += subset_image_count * subset.get("num_repeats", 1)

    import math
    steps_per_epoch = math.ceil(total_images_with_repeats / train_batch_size)
    calculated_max_steps = math.ceil(steps_per_epoch / gradient_accumulation_steps) * max_train_epochs

  num_cycles = lr_scheduler_num_cycles or 1
  cycle_steps = calculated_max_steps // num_cycles
  lr_scheduler_args_list.append(f"first_cycle_max_steps={cycle_steps}")
  warmup_steps = round(calculated_max_steps * lr_warmup_ratio) // num_cycles
  if warmup_steps > 0:
    lr_scheduler_args_list.append(f"warmup_steps={warmup_steps}")

def main():
  if not os.path.exists('/content/drive'):
    try:
      from google.colab import drive
      print("📂 Conectando a Google Drive...")
      drive.mount('/content/drive')
    except:
      pass

  for d in (main_dir, log_folder, images_folder, output_folder, config_folder):
    os.makedirs(d, exist_ok=True)

  if not validate_dataset():
    return

  install_trainer()

  if not download_anima_components():
    print("💥 Error descargando componentes de Anima.")
    return

  if lr_scheduler_type:
    create_config()
    os.chdir(kohya_dir)
    calculate_rex_steps()
    os.chdir(root_dir)

  create_config()

  print("\n ⭐ Iniciando Entrenador Anima DiT LoRA..\n ")
  os.chdir(kohya_dir)
  os.environ['NUMEXPR_NUM_THREADS'] = '2'
  os.environ["TERM"] = "dumb"
  _run_cmd(f"{venv_python} {train_network} --console_log_simple --config_file={config_file} --dataset_config={dataset_config_file}")
  os.chdir(root_dir)

  display(Markdown("### ✅ ¡Terminado! Tus archivos de LoRA están en la carpeta de output."))

main()

if auto_off:
    print("\033[96mAuto Off encendido, el entorno se desconectara en 30 segundos\033[0m")
    try:
        from google.colab import runtime
        time.sleep(30)
        runtime.unassign()
    except:
        pass


In [ ]:
#@markdown ##Conectarse a Google Drive 📁 <p> Algunas de las opciones de abajo necesitan conexión a Google Drive
#@markdown conectate aquí.
from google.colab import drive
drive.mount('/content/drive')
print("Conexión establecida a Drive Exitosa!")

In [ ]:
#@markdown ### ↪️ Continuar

#@markdown Aquí puede escribir una ruta en su Google Drive para cargar un archivo Lora existente y continuar entrenando. <P>
#@markdown **Advertencia:** No es lo mismo que una larga sesión de entrenamiento. Los epochs comienzan desde cero, y puede obtener peores resultados.
continue_from_lora = "" #@param {type:"string"}
if continue_from_lora and not continue_from_lora.startswith("/content/drive/MyDrive"):
  import os
  continue_from_lora = os.path.join("/content/drive/MyDrive", continue_from_lora)

### 📚 Varias carpetas en el mismo conjunto de datos
A continuación se muestra una plantilla que le permite definir varias carpetas en su conjunto de datos. Debe incluir la ubicación de cada carpeta y puede establecer un número diferente de repeticiones para cada una. Para agregar más carpetas, simplemente copie y pegue las secciones que comienzan con `[[datasets.subsets]]`.

Al habilitar esto, se ignorará el número de repeticiones establecidas en la celda principal y también se ignorará la carpeta principal establecida por el nombre del proyecto.

Puede convertir uno de ellos en una carpeta de regularización agregando `is_reg = true`  
También puede establecer diferentes `keep_tokens`, `flip_aug`, etc.

In [ ]:
custom_dataset = """
[[datasets]]

[[datasets.subsets]]
image_dir = "/content/drive/MyDrive/Loras/example/dataset/good_images"
num_repeats = 3

[[datasets.subsets]]
image_dir = "/content/drive/MyDrive/Loras/example/dataset/normal_images"
num_repeats = 1

"""

In [ ]:
custom_dataset = None

##Extras

In [ ]:
#@markdown ## Colab Activo en el celular 📳
#@markdown Ejecuta esta celda para mantener viva la pestaña en el celular (antes de ejecutar la celda de inicio) <p>
#@markdown puedes convinarlo con esta [extensión](https://chromewebstore.google.com/detail/google-colab-keep-alive/bokldcdphgknojlbfhpbbgkggjfhhaek) en tu celular o pc (en celular necesitas un navegador con soporte a extensiones, `firefox` no funcionara)
%%html
<b>Pulsa play en el reproductor de música para mantener viva la pestaña antes de ejecutar la celda de inicio (sólo utiliza 13 MB de datos).</b><br/>
<audio src="https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a" controls>

In [ ]:
#@markdown ### 🔢 Contar conjuntos de datos
#@markdown Google Drive hace imposible contar los archivos en una carpeta, por lo que le mostrará el recuento de archivos en todas las carpetas y subcarpetas.
folder = "/content/drive/MyDrive/Loras/ejemplo/dataset" #@param {type:"string"}

import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    print("📂 Connecting to Google Drive...\n")
    drive.mount('/content/drive')

tree = {}
exclude = ("_logs", "/output")
for i, (root, dirs, files) in enumerate(os.walk(folder, topdown=True)):
  dirs[:] = [d for d in dirs if all(ex not in d for ex in exclude)]
  images = len([f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))])
  captions = len([f for f in files if f.lower().endswith(".txt")])
  others = len(files) - images - captions
  path = root[folder.rfind("/")+1:]
  tree[path] = None if not images else f"{images:>4} images | {captions:>4} captions |"
  if tree[path] and others:
    tree[path] += f" {others:>4} other files"

pad = max(len(k) for k in tree)
print("\n".join(f"📁{k.ljust(pad)} | {v}" for k, v in tree.items() if v))

In [ ]:
#@markdown ##Calculador de Repeticiones ⌛📝
#@markdown Calcula el número de repeticiones a usar para entrenar tu lora, Recuerda que en `SDXL y Pony` se usa un batch de `4`.
#@markdown Si usas colab pro calcula tus repeticiones con `8` de batch size
# Define las Variables
# Número de imágenes
num_images = 21 # @param{type:"number"}
# Número de repeticiones
num_repeats = 2 # @param{type:"number"}
# Número de epocas
num_epochs = 70 # @param{type:"number"}
# Tamaño de lote
batch_size = 2 # @param{type:"number"}

# Calcula el resultado
resultado = (num_images * num_repeats * num_epochs) / batch_size

# Muestra el resultado
print("\33[96mEl total de repeticiones es:\033[0m", resultado)

In [ ]:
#@markdown ### 📂 Descomprimir conjunto de datos
#@markdown Es mucho más lento cargar archivos individuales en drive, por lo que es posible que desee cargar un archivo zip si tiene su conjunto de datos en su computadra, aqui puedes descomprimirlo.
zip = "/content/drive/MyDrive/my_dataset.zip" #@param {type:"string"}
extract_to = "/content/drive/MyDrive/Loras/example/dataset" #@param {type:"string"}

import os, zipfile

if not os.path.exists('/content/drive'):
  from google.colab import drive
  print("📂 Connecting to Google Drive...")
  drive.mount('/content/drive')

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip, 'r') as f:
  f.extractall(extract_to)

print("✅ Done")

## Iniciar sesión y crear repositorio en huggingFace 🤗

In [ ]:
from huggingface_hub import login
from huggingface_hub import HfApi
from huggingface_hub.utils import validate_repo_id, HfHubHTTPError

# @markdown crea una cuenta de HuggingFace [aquí](https://huggingface.co/) o si ya la tienes simplemente usa tu token 👇
# @markdown > Obten **tú** token `EN WRITE` de Huggingface [aquí](https://huggingface.co/settings/tokens) 👈
write_token = ""  # @param {type:"string"}
#@markdown Complete esto si desea cargarlo en su organización, o simplemente déjelo vacío.
orgs_name = ""  # @param{type:"string"}
# @markdown Si su repositorio de modelo/conjunto de datos no existe, lo creará automáticamente.
#@markdown No se permiten espacios usa `un guión bajo` para nombres largos

#@markdown El `model_name` se usa para subir archivos individuales
model_name = ""  # @param{type:"string"}
#@markdown El dataset se usa para subir carpetas enteras.
dataset_name = "loras"  # @param{type:"string"}
#@markdown Marca esta casilla si quieres que tu repositorio/dataset sea privado de lo contrario sera publico
make_private = True  # @param{type:"boolean"}

def authenticate(write_token):
    login(write_token, add_to_git_credential=True)
    api = HfApi()
    return api.whoami(write_token), api


def create_repo(api, user, orgs_name, repo_name, repo_type, make_private=False):
    global model_repo
    global datasets_repo

    if orgs_name == "":
        repo_id = user["name"] + "/" + repo_name.strip()
    else:
        repo_id = orgs_name + "/" + repo_name.strip()

    try:
        validate_repo_id(repo_id)
        api.create_repo(repo_id=repo_id, repo_type=repo_type, private=make_private)
        print(f"{repo_type.capitalize()} repo '{repo_id}' didn't exist, creating repo")
    except HfHubHTTPError as e:
        print(f"{repo_type.capitalize()} repo '{repo_id}' exists, skipping create repo")

    if repo_type == "model":
        model_repo = repo_id
        print(f"{repo_type.capitalize()} repo '{repo_id}' link: https://huggingface.co/{repo_id}\n")
    else:
        datasets_repo = repo_id
        print(f"{repo_type.capitalize()} repo '{repo_id}' link: https://huggingface.co/datasets/{repo_id}\n")

user, api = authenticate(write_token)

if model_name:
    create_repo(api, user, orgs_name, model_name, "model", make_private)
if dataset_name:
    create_repo(api, user, orgs_name, dataset_name, "dataset", make_private)

##Subir a huggingface_hub (Lora y dataset) 🤗

In [ ]:
#@markdown ##subir lora (safetensors)
from huggingface_hub import HfApi
from pathlib import Path

api = HfApi()
file_path = "/content/drive/MyDrive/Loras/Mi_proyecto/output/Mi_lora.safetensors" #@param {type :"string"}
#@markdown Comentario (Opcional) puede ser util para identificar versiones del lora, `ejemplo: pony, animagine, V1, V2 etc...`
commit_message = ""  #@param {type :"string"}

if file_path != "":
  path_obj = Path(file_path)
  trained_model = path_obj.parts[-1]

  print(f"Uploading {trained_model} to https://huggingface.co/"+model_repo)
  print(f"Please wait...")

  api.upload_file(
      path_or_fileobj=file_path,
      path_in_repo=trained_model,
      repo_id=model_repo,
      commit_message=commit_message,
  )

  print(f"Upload success, located at https://huggingface.co/"+model_repo+"/blob/main/"+trained_model+"\n")
print('¡¡Subida finalizada!!')

In [ ]:
# @markdown ## Subir carpeta 📁 a HuggingFace 🤗
from huggingface_hub import HfApi
from pathlib import Path
import os

api = HfApi()

# @markdown Puedes subir una carpeta entera a Huggingface con tus loras entrenados, incluso puedes subir toda la carpeta de tu proyecto (incluido sub-carpetas), ten paciencia podria tardar un tiempo en cargar varios archivos.
folder_path = "/content/drive/MyDrive/Loras/Mi_lora/output"  # @param {type :"string"}

# @markdown Nombre personalizado para la carpeta a subir
folder_name = "Mi_lora"  # @param {type :"string"}

# @markdown Comentario (Opcional)
commit_message = ""  # @param {type :"string"}

if not commit_message:
    commit_message = "feat: upload folder"

def upload_folder(folder_path, folder_name):
    print(f"Uploading {folder_name} to https://huggingface.co/datasets/{datasets_repo}")
    print("Please wait...")

    api.upload_folder(
        folder_path=folder_path,
        path_in_repo=folder_name,
        repo_id=datasets_repo,
        repo_type="dataset",
        commit_message=commit_message,
        ignore_patterns=".ipynb_checkpoints",
    )
    print(f"Carga terminada, puedes encontrarlo en https://huggingface.co/datasets/{datasets_repo}/tree/main/{folder_name}\n")

def upload():
    if folder_path:
        upload_folder(folder_path, folder_name)

upload()

# 📈 Gráficas del entrenamiento
Puedes hacer esto después de ejecutar el entrenador.  No necesitas esto a menos que sepas lo que estás haciendo.  
 Es posible que la primera celda a continuación no cargue todos sus registros.  Continúe probando la segunda celda hasta que se hayan cargado todos los datos.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir={log_folder}/

In [ ]:
from tensorboard import notebook
notebook.display(port=6006, height=800)